# 02L — Reading-frame rule


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📋 frame with phenotype table


In [3]:
tab = pd.crosstab(d['frame'], d['phenotype'])
display(tab)


phenotype,BMD,DMD
frame,,
in-frame,423,2344
out-of-frame,263,2833


## 2. 📊 counts plot 🛡


In [4]:
tmp = d[['frame', 'phenotype']].dropna()
fig = px.histogram(tmp, x='frame', color='phenotype', barmode='stack', title='Reading-frame rule counts')
fig.show()


## 3. 📊 proportion plot 🛡


In [5]:
tab = pd.crosstab(d['frame'], d['phenotype'], normalize='index')
fig = px.bar(tab.reset_index().melt(id_vars='frame', var_name='phenotype', value_name='fraction'), x='frame', y='fraction', color='phenotype', barmode='stack', title='Reading-frame rule proportions')
fig.show()


## 4. 🧪 χ² frame ~ phenotype ⭐


In [6]:
tab, chi2, p, dof = chi2_table(d, 'frame', 'phenotype')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


phenotype,BMD,DMD
frame,,
in-frame,423,2344
out-of-frame,263,2833


,value
chi2,6.459485e+01
dof,1.000000e+00
p_value,9.199566e-16


## 5. 🧪 Fisher DMD vs BMD ⭐


In [7]:
tmp = d[d['phenotype'].isin(['DMD', 'BMD'])][['phenotype', 'frame']].dropna().copy()
tmp['is_dmd'] = tmp['phenotype'].eq('DMD')
tmp['is_out_of_frame'] = tmp['frame'].eq('out-of-frame')
tab, odds, p, ci = fisher_bool(tmp, 'is_dmd', 'is_out_of_frame')
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


is_out_of_frame,False,True
is_dmd,,
False,423,263
True,2344,2833


,value
odds_ratio,1.943899e+00
p_value,7.282618e-16
or_ci_low,1.650939e+00
or_ci_high,2.288843e+00


## 6. 📋 odds ratio ⭐


In [8]:
tmp = d[d['phenotype'].isin(['DMD', 'BMD'])][['phenotype', 'frame']].dropna().copy()
tmp['is_dmd'] = tmp['phenotype'].eq('DMD')
tmp['is_out_of_frame'] = tmp['frame'].eq('out-of-frame')
tab = pd.crosstab(tmp['is_dmd'], tmp['is_out_of_frame']).reindex(index=[False, True], columns=[False, True], fill_value=0)
a = int(tab.loc[True, True])
b = int(tab.loc[True, False])
c = int(tab.loc[False, True])
dd = int(tab.loc[False, False])
or_value, low, high = odds_ratio_ci(a, b, c, dd)
display(pd.Series({'odds_ratio': or_value, 'ci_low': low, 'ci_high': high, 'a_DMD_out': a, 'b_DMD_in': b, 'c_BMD_out': c, 'd_BMD_in': dd}).to_frame('value'))


,value
odds_ratio,1.943899
ci_low,1.650939
ci_high,2.288843
a_DMD_out,2833.000000
b_DMD_in,2344.000000
c_BMD_out,263.000000
d_BMD_in,423.000000


## 7. 📊 out-of-frame fractions


In [9]:
tbl = d.groupby('phenotype').agg(out_of_frame_fraction=('frame', lambda s: (s == 'out-of-frame').mean()), n=('var_id', 'size')).reset_index().sort_values('n', ascending=False)
display(tbl)
fig = px.bar(tbl, x='phenotype', y='out_of_frame_fraction', hover_data=['n'], title='Out-of-frame fraction by phenotype')
fig.show()


,phenotype,out_of_frame_fraction,n
1,DMD,0.362879,7807
0,BMD,0.257087,1023


## 8. 🧪 z-test ⭐


In [10]:
tmp = d[d['phenotype'].isin(['DMD', 'BMD'])][['phenotype', 'frame']].dropna().copy()
n1 = (tmp['phenotype'] == 'DMD').sum()
n2 = (tmp['phenotype'] == 'BMD').sum()
x1 = ((tmp['phenotype'] == 'DMD') & (tmp['frame'] == 'out-of-frame')).sum()
x2 = ((tmp['phenotype'] == 'BMD') & (tmp['frame'] == 'out-of-frame')).sum()
p_pool = (x1 + x2) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
z = (x1 / n1 - x2 / n2) / se
p = 2 * (1 - norm.cdf(abs(z)))
display(pd.Series({'n_DMD': n1, 'n_BMD': n2, 'out_DMD': x1, 'out_BMD': x2, 'z_stat': z, 'p_value': p}).to_frame('value'))


,value
n_DMD,5.177000e+03
n_BMD,6.860000e+02
out_DMD,2.833000e+03
out_BMD,2.630000e+02
z_stat,8.077787e+00
p_value,6.661338e-16


## 9. 📊 frame with mutation_type


In [11]:
heat = pd.crosstab(d['frame'], d['mutation_type'])
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Blues', title='Frame x mutation_type')
fig.show()


## 10. 🧪 χ²


In [12]:
tab, chi2, p, dof = chi2_table(d, 'frame', 'mutation_type')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


mutation_type,frameshift,large_deletion,missense,nonsense,splice
frame,,,,,
in-frame,0,0,3328,0,0
out-of-frame,754,1701,0,791,940


,value
chi2,7514.0
dof,4.0
p_value,0.0


## 11. 📊 exception plot 📖


In [18]:
tmp = d[['phenotype', 'frame']].dropna().copy()
tmp['is_exception'] = ((tmp['phenotype'] == 'DMD') & (tmp['frame'] == 'in-frame')) | ((tmp['phenotype'] == 'BMD') & (tmp['frame'] == 'out-of-frame'))
fig = px.histogram(tmp, x='phenotype', color='is_exception', barmode='stack', title='Reading-frame exceptions by phenotype')
fig.show()


## 12. 📋 exception table 📖


In [19]:
tmp = d[['var_id', 'phenotype', 'frame', 'mutation_type', 'domain_clean', 'exon_num']].dropna(subset=['phenotype', 'frame']).copy()
tmp['is_exception'] = ((tmp['phenotype'] == 'DMD') & (tmp['frame'] == 'in-frame')) | ((tmp['phenotype'] == 'BMD') & (tmp['frame'] == 'out-of-frame'))
display(tmp[tmp['is_exception']].head(50))
display(pd.Series({'n_exceptions': int(tmp['is_exception'].sum())}).to_frame('value'))


,var_id,phenotype,frame,mutation_type,domain_clean,exon_num,is_exception
142,4065957,BMD,out-of-frame,frameshift,NaN,79.0,True
155,583255,DMD,in-frame,missense,Disordered,79.0,True
156,2769815,DMD,in-frame,missense,Spectrin 2,79.0,True
157,2703886,DMD,in-frame,missense,Spectrin 2,79.0,True
158,1160748,DMD,in-frame,missense,Spectrin 2,79.0,True
187,2674873,BMD,out-of-frame,splice,NaN,78.0,True
199,2061097,DMD,in-frame,missense,Disordered,78.0,True
201,94450,DMD,in-frame,missense,Disordered,78.0,True
203,1001377,DMD,in-frame,missense,Disordered,78.0,True
209,952891,BMD,out-of-frame,splice,NaN,77.0,True


,value
n_exceptions,2607


## 13. 📊 exception vs domain 📖


In [15]:
tmp = d[['phenotype', 'frame', 'domain_clean']].dropna().copy()
tmp['is_exception'] = ((tmp['phenotype'] == 'DMD') & (tmp['frame'] == 'in-frame')) | ((tmp['phenotype'] == 'BMD') & (tmp['frame'] == 'out-of-frame'))
top = tmp['domain_clean'].value_counts().head(15).index
heat = pd.crosstab(tmp[tmp['domain_clean'].isin(top)]['domain_clean'], tmp[tmp['domain_clean'].isin(top)]['is_exception'])
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Reds', title='Exceptions vs domain')
fig.show()


## 14. 🧪 Fisher exceptions in rod domain 📖


In [16]:
tmp = d[['phenotype', 'frame', 'domain_clean']].dropna().copy()
tmp['is_exception'] = ((tmp['phenotype'] == 'DMD') & (tmp['frame'] == 'in-frame')) | ((tmp['phenotype'] == 'BMD') & (tmp['frame'] == 'out-of-frame'))
tmp['is_rod'] = tmp['domain_clean'].str.contains('spectrin', case=False, na=False)
tab, odds, p, ci = fisher_bool(tmp, 'is_rod', 'is_exception')
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


is_exception,False,True
is_rod,,
False,221,341
True,752,1447


,value
odds_ratio,1.247064
p_value,0.025941
or_ci_low,1.030426
or_ci_high,1.509247


## 15. 📊 exception vs exon


In [17]:
tmp = d[['phenotype', 'frame', 'exon_num']].dropna().copy()
tmp['is_exception'] = ((tmp['phenotype'] == 'DMD') & (tmp['frame'] == 'in-frame')) | ((tmp['phenotype'] == 'BMD') & (tmp['frame'] == 'out-of-frame'))
tmp['exon'] = tmp['exon_num'].astype(int)
tbl = tmp.groupby('exon').agg(exception_fraction=('is_exception', 'mean'), n=('is_exception', 'size')).reset_index()
fig = px.scatter(tbl, x='exon', y='exception_fraction', size='n', title='Exception fraction by exon')
fig.show()
